# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [2]:
from _imports import * # Centralized file containing all imports

2025-06-14 21:37:01.356321: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-14 21:37:01.366664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749947821.378406   74813 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749947821.381882   74813 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-14 21:37:01.393676: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [3]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()


TensorFlow GPU Monitor - 2025-06-14 21:37:02
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9
GPUs Detected  : 1
Default Device : /device:GPU:0

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3050      0.1GB /    4.0GB  40C    23%   

Memory Summary
Total GPU Memory :      4.0 GB
Used Memory      :      0.1 GB (  3.2%)
Free Memory      :      3.9 GB ( 96.8%)



2025-06-14 21:37:02.613121: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1749947822.613156   74813 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1749947822.613310   74813 gpu_device.cc:2022] Created device /device:GPU:0 with 2229 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## 2. Run Parameters 

In [4]:
NUM_TRIALS = 1
EPOCHS = 1

SEED = 812

In [5]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

In [6]:
TOP_K = 10 # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = True  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

In [7]:
# Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_cnn1d_flat_v9_1"  # None or "runs/nas_1"

RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_")

## 3. Data Loading and Preprocessing

In [8]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/media/matheus/SSD/Projects/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)


/media/matheus/SSD/Projects/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)


/media/matheus/SSD/Projects/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


In [9]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Getters

### 4.1. Callbacks

In [10]:
def get_callbacks(trial: optuna.Trial, backup_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        backup_dir (str): Directory where the backup files will be stored.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=10,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=5,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )
    
    # Backup and restore the model
    # backup = callbacks.BackupAndRestore(backup_dir=backup_dir)
    
    # Model checkpointing
    # checkpoint = callbacks.ModelCheckpoint(
    #     filepath=os.path.join(backup_dir, "checkpoint.h5"),
    #     monitor=monitor,
    #     save_best_only=True,
    #     save_weights_only=True,
    # )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = callbacks.TerminateOnNaN()

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor, interval=5)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, nan_pruner_callback, pruning_callback]


def get_activation(function: str) -> tf.keras.layers.Layer:
    """
    Returns the activation layer based on the provided function name.
    
    Args:
        function (str): Name of the activation function.
        
    Returns:
        tf.keras.layers.Layer: Corresponding activation layer.
    """
    if function == "relu":
        return layers.Activation("relu")
    elif function == "tanh":
        return layers.Activation("tanh")
    elif function == "sigmoid":
        return layers.Activation("sigmoid")
    elif function == "swish":
        return layers.Activation("swish")
    else:
        raise ValueError(f"Unsupported activation function: {function}")

## 5. Hyperparameters

In [11]:
hparams = HParams(
    activation_choices=[
        #? ReLU family
        "relu",
        # "leaky_relu",
        # "elu",
        # "celu",
        # "selu",
        #? Smooth ReLU-like and modern variants
        # "softplus",
        # "gelu",
        # "mish",
        "swish",
        # "hard_silu",
        #? Tanh family
        "tanh",
        # "hard_tanh",
        # "softsign",
        #? Sigmoid family
        "sigmoid",
        # "hard_sigmoid",
        # "log_sigmoid",
        #? Linear and Exponential
        # "linear",
        # "exponential",
        #? Gated and transformer-related
        # "glu",
        # "softmax",
        #? Sparsity and uncommon
        # "hard_shrink",
    ],
    regularizer_choices=[
        "none",
        "l1",
        "l2",
        "l1l2",
    ],
    optimizer_choices=[
        # "SGD",
        # "RMSprop",
        # "Adam",
        # "AdamW",
        # "Adadelta",
        # "Adagrad",
        # "Adamax",
        # "Adafactor",
        # "Nadam",
        # "Ftrl",
        "Lion",
        # "Lamb",
        # "LossScaleOptimizer",
    ],
    scaler_choices=[
        "StandardScaler",
        "MinMaxScaler_0_1",
        "MinMaxScaler_-1_1",
        "RobustScaler",
        "QuantileTransformer",
        "PowerTransformer",
    ],
    l1_value=1e-2,
    l2_value=1e-2,
    min_lr=8e-5,
    max_lr=2e-4,
)

initializer_options = [
    initializers.Zeros(),
    initializers.Ones(),
    initializers.Constant(),
    initializers.RandomNormal(),
    initializers.RandomUniform(),
    initializers.TruncatedNormal(),
    initializers.GlorotNormal(),
    initializers.GlorotUniform(),
    initializers.HeNormal(),
    initializers.HeUniform(),
    initializers.LecunNormal(),
    initializers.LecunUniform(),
    initializers.Identity(),
    initializers.Orthogonal(),
    initializers.VarianceScaling(),
]

## 6. Objective Function

In [12]:
def objective(
    trial: optuna.Trial,
    backup_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    history_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    show_summary: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        backup_dir (str): Path to store backup files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        history_dir (str): Path to store training history.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        show_summary (bool): If True, display the model summary.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    # Redundancy cleanup
    (clear_session(), gc.collect())

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global x_lidar_train
    global x_coord_train
    global y_train
    global x_lidar_val
    global x_coord_val
    global y_val
    global x_lidar_test
    global x_coord_test
    global y_test
    global s009_lidar_input
    global s009_coord_input
    global s009_y

    # Each trial gets a different seed
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————— Data Preprocessing ———————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
        x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

        # Inline one-hot encoding of semantic values
        one_hot_lidar = layers.Lambda(
            lambda x: tf.concat(
                [
                    # “Is there a BS anywhere in the 10 channels?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                    # “Vehicle?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                    # “Obstacle?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                    # “Free?” → 1 channel (all channels zero)
                    tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
                ],
                axis=-1,
            ),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20, 200, 4),
            name="lidar_transform_to_one_hot",
        )(x_lidar_input)
        # -> (batch, 20, 200, 4)

        # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
        x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(
            one_hot_lidar
        )

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(2,), name="coord_input")

        # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
        x_coord: layers.Layer = layers.Lambda(
            lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20 * 200, 2),
            name="coord_tile_flat",
        )(x_coord_input)

        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
        combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

        # ———————————————————————————————— Conv Layers ——————————————————————————————— #
        # First Conv1D layer
        x = build_cnn1d(
            trial=trial,
            hparams=hparams,
            x=combined,
            name_prefix="conv1d_0",
            # Filters
            filters_range=(400, 600),
            filters_step=40,
            # Kernel size
            kernel_size_range=(3, 12),
            kernel_size_step=3,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
        )
        pool_size = trial.suggest_categorical("pool_size_0", [3, 4, 5])
        x = layers.MaxPooling1D(pool_size=pool_size, name="max_pool_0")(x)

        # Second Conv1D layer
        x = build_cnn1d(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="conv1d_1",
            # Filters
            filters_range=(200, 400),
            filters_step=40,
            # Kernel size
            kernel_size_range=(3, 12),
            kernel_size_step=3,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_1_strides", 1, 2),
        )
        pool_size = trial.suggest_categorical("pool_size_1", [3, 4, 5])
        x = layers.MaxPooling1D(pool_size=pool_size, name="max_pool_1")(x)

        # Third Conv1D layer
        x = build_cnn1d(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="conv1d_2",
            # Filters
            filters_range=(400, 600),
            filters_step=40,
            # Kernel size
            kernel_size_range=(3, 12),
            kernel_size_step=3,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_2_strides", 1, 2),
        )
        pool_size = trial.suggest_categorical("pool_size_2", [3, 4, 5])
        x = layers.MaxPooling1D(pool_size=pool_size, name="max_pool_2")(x)

        # Fourth Conv1D layer
        x = build_cnn1d(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="conv1d_3",
            # Filters
            filters_range=(50, 150),
            filters_step=20,
            # Kernel size
            kernel_size_range=(3, 12),
            kernel_size_step=3,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_3_strides", 1, 2),
        )
        pool_size = trial.suggest_categorical("pool_size_3", [3, 4, 5])
        x = layers.MaxPooling1D(pool_size=pool_size, name="max_pool_3")(x)

        # ———————————————————————————— Extra dense layers ———————————————————————————— #
        # (batch, length, channels) -> (batch, length * channels)
        x = layers.Flatten(name="flatten")(x)

        x = build_dnn(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="dense_1",
            # Units
            units_range=(150, 400),
            units_step=50,
            # Dropout
            dropout_rate_range=(0.0, 0.3),
            dropout_rate_step=0.1,
        )

        x = build_dnn(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="dense_2",
            # Units
            units_range=(150, 400),
            units_step=50,
            # Dropout
            dropout_rate_range=(0.0, 0.3),
            dropout_rate_step=0.1,
        )

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="softmax", name="output")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                Train the Model                               #
        # ———————————————————————————————————————————————————————————————————————————— #
        model.summary() if show_summary else None

        model.compile(
            optimizer=hparams.get_optimizer(trial),
            loss=losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        batch_size = 64
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, backup_dir),
            verbose=1,
        )

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])

        if size_penalizer == "flops":
            loss = compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = compute_params_penalized_loss(loss=loss, model=model, params_penalty_factor=1e-8)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————————— Plot Results ——————————————————————————————— #
        # Configure axis
        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])
        val_loss_best = min(history.history["val_loss"])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ———————————————————————— Save model characteristics ———————————————————————— #
        params = model.count_params()
        bits_per_param = tf.dtypes.as_dtype(POLICY.variable_dtype).size

        trial.set_user_attr("num_params", params)
        trial.set_user_attr("model_size", params * bits_per_param)
        trial.set_user_attr("flops", get_flops(model))
        trial.set_user_attr("macs", get_macs(model))
        trial.set_user_attr("model_summary", capture_model_summary(model))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))

        # ————————————————————————————— Print the results ———————————————————————————— #
        clear()

        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "=" * 15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {val_loss_best:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009): {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}\n")
        print(f"Test loss (s009 full): {test_loss_full:.12f}")
        print(f"Test accuracy (s009 full): {test_acc_full:.4f}\n")

        params = model.count_params()
        print(f"Number of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ———————————————————————————————————————————————————————————————————————————— #
        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        print(f"\n❌ Trial {trial.number} hit OOM (Resource Exhausted)\n")
        with open(os.path.join(logs_dir, f"oom_trials.log"), "a") as f:
            f.write(f"OOM error during trial {trial.number}:\n{traceback.format_exc()}\n\n")

        return float("inf")  # Return bad loss
    except Exception as e:
        with open(os.path.join(logs_dir, f"error_trial_{trial.number}.log"), "w") as f:
            f.write(f"An error occurred during trial:\n{e}\n{traceback.format_exc()}\n\n")

        raise  # Re-raise the exception to propagate it
    finally:
        for v in [
            "model",
            "history",
            "history_df",
        ]:
            if v in globals() and globals()[v] is not None:
                del globals()[v]
        plt.close("all")
        clear_session()
        gc.collect()

## 7. Code Health Check

In [13]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# log_resources(log_dir=resources_dir)

## Main

In [14]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    # Initialize directories for the study
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
    ) = init_study_dirs(RUN_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        direction="maximize",
        pruner=optuna.pruners.HyperbandPruner(),
        load_if_exists=True,
    )

    study.optimize(
        lambda trial: objective(
            trial,
            backup_dir=backup_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            history_dir=history_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            show_summary=True,
        ),
        n_trials=get_remaining_trials(study, NUM_TRIALS),
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,
        rank_descending=RANK_DESCENDING,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # ————————————————————————————— Log Trial Results ———————————————————————————— #
    with open(f"{study_dir}/trials.log", "w") as f:
        f.write(
            f"Total trials: {len(study.trials)}\n"
            f"Pruned trials: {sum(t.state==TrialState.PRUNED for t in study.trials)}\n"
            f"Failed trials: {sum(t.state==TrialState.FAIL for t in study.trials)}\n"
        )

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (clear(), analyze_study(study, table_dir=os.path.join(study_dir, "analysis")))
    
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Write success flag for the auto restart script
    Path("/tmp/success.flag").write_text("SUCCESS")

    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)



Analyzing study...
Generating summary tables...
Creating hyperparameter distribution plots...
Creating numeric parameters distribution plot (17 parameters)...

 An error occurred: `dataset` input should have multiple elements.



Traceback (most recent call last):
  File "/tmp/ipykernel_74813/4123217264.py", line 90, in <module>
    (clear(), analyze_study(study, table_dir=os.path.join(study_dir, "analysis")))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/matheus/SSD/Projects/RayWise/src/araras/optuna/analyze.py", line 1290, in analyze_study
    plot_hyperparameter_distributions(df, numeric_cols, categorical_cols, dirs)
  File "/media/matheus/SSD/Projects/RayWise/src/araras/optuna/analyze.py", line 363, in plot_hyperparameter_distributions
    kde = gaussian_kde(values)
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/scipy/stats/_kde.py", line 199, in __init__
    raise ValueError("`dataset` input should have multiple elements.")
ValueError: `dataset` input should have multiple elements.
